# Risk Factor Performance Analysis

This notebook analyzes how different risk factor combinations affect game performance.

In [1]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

# Load data
stats_dir = project_root / "stats"
analysis_files = list(stats_dir.glob("analysis_*.json")) if stats_dir.exists() else []

if analysis_files:
    latest_file = max(analysis_files, key=lambda p: p.stat().st_mtime)
    with open(latest_file, 'r') as f:
        data = json.load(f)
    print(f"Loaded: {latest_file.name}")
else:
    print("No data files found")
    data = None

Loaded: analysis_20251123_134629.json


## Risk Factor Impact Analysis

In [2]:
if data:
    win_rates = data.get('win_rates_by_combination', {})
    
    if win_rates:
        print("=" * 60)
        print("RISK FACTOR PERFORMANCE ANALYSIS")
        print("=" * 60)
        
        # Convert to DataFrame for analysis
        risk_data = []
        for risk_key, rates in win_rates.items():
            # Parse risk key: ((trump0, gameplay0), (trump1, gameplay1), (trump2, gameplay2), (trump3, gameplay3))
            if isinstance(risk_key, str):
                # Handle string representation
                import ast
                try:
                    risk_key = ast.literal_eval(risk_key)
                except:
                    continue
            
            team0_risk = risk_key[0] if len(risk_key) > 0 else (0.5, 0.5)
            team1_risk = risk_key[2] if len(risk_key) > 2 else (0.5, 0.5)
            
            risk_data.append({
                'risk_combination': str(risk_key),
                'team0_trump_risk': team0_risk[0],
                'team0_gameplay_risk': team0_risk[1],
                'team1_trump_risk': team1_risk[0],
                'team1_gameplay_risk': team1_risk[1],
                'team0_win_rate': rates.get('team0_rate', 0),
                'team1_win_rate': rates.get('team1_rate', 0),
                'total_games': rates.get('total_games', 0),
            })
        
        df_risk = pd.DataFrame(risk_data)
        
        # Average win rate by risk level
        print("\n" + "=" * 60)
        print("AVERAGE WIN RATE BY RISK LEVEL")
        print("=" * 60)
        
        for team in [0, 1]:
            print(f"\nTeam {team}:")
            for risk_type in ['trump', 'gameplay']:
                risk_col = f'team{team}_{risk_type}_risk'
                win_col = f'team{team}_win_rate'
                
                # Group by risk level bins
                df_risk[f'{risk_type}_bin'] = pd.cut(df_risk[risk_col], bins=[0, 0.25, 0.5, 0.75, 1.0], labels=['Low (0-0.25)', 'Medium (0.25-0.5)', 'High (0.5-0.75)', 'Very High (0.75-1.0)'])
                
                grouped = df_risk.groupby(f'{risk_type}_bin')[win_col].agg(['mean', 'count'])
                print(f"  {risk_type.capitalize()} Risk:")
                for bin_name, row in grouped.iterrows():
                    if pd.notna(bin_name):
                        print(f"    {bin_name}: {row['mean']:.3f} ({int(row['count'])} combinations)")
        
        # Best performing combinations
        print("\n" + "=" * 60)
        print("BEST PERFORMING COMBINATIONS")
        print("=" * 60)
        
        print("\nTop 10 Team 0 Win Rates:")
        top_team0 = df_risk.nlargest(10, 'team0_win_rate')[['risk_combination', 'team0_win_rate', 'total_games']]
        print(top_team0.to_string(index=False))
        
        print("\nTop 10 Team 1 Win Rates:")
        top_team1 = df_risk.nlargest(10, 'team1_win_rate')[['risk_combination', 'team1_win_rate', 'total_games']]
        print(top_team1.to_string(index=False))
        
        # Team composition effects
        print("\n" + "=" * 60)
        print("TEAM COMPOSITION EFFECTS")
        print("=" * 60)
        
        # Analyze balanced vs unbalanced teams
        df_risk['team0_avg_risk'] = (df_risk['team0_trump_risk'] + df_risk['team0_gameplay_risk']) / 2
        df_risk['team1_avg_risk'] = (df_risk['team1_trump_risk'] + df_risk['team1_gameplay_risk']) / 2
        df_risk['risk_difference'] = abs(df_risk['team0_avg_risk'] - df_risk['team1_avg_risk'])
        
        balanced = df_risk[df_risk['risk_difference'] < 0.2]
        unbalanced = df_risk[df_risk['risk_difference'] >= 0.2]
        
        print(f"\nBalanced Teams (risk diff < 0.2): {len(balanced)} combinations")
        if len(balanced) > 0:
            print(f"  Team 0 avg win rate: {balanced['team0_win_rate'].mean():.3f}")
            print(f"  Team 1 avg win rate: {balanced['team1_win_rate'].mean():.3f}")
        
        print(f"\nUnbalanced Teams (risk diff >= 0.2): {len(unbalanced)} combinations")
        if len(unbalanced) > 0:
            print(f"  Team 0 avg win rate: {unbalanced['team0_win_rate'].mean():.3f}")
            print(f"  Team 1 avg win rate: {unbalanced['team1_win_rate'].mean():.3f}")
        
        # Store for visualization
        risk_analysis_data = df_risk
    else:
        print("No win rate data found")
        risk_analysis_data = None
else:
    print("No data loaded")
    risk_analysis_data = None

No win rate data found


## Visualize Risk Factor Performance

In [3]:
if risk_analysis_data is not None and len(risk_analysis_data) > 0:
    # Set style
    plt.style.use('seaborn-v0_8')
    sns.set_palette("husl")
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Win rate vs Trump Risk
    ax1 = axes[0, 0]
    for team in [0, 1]:
        risk_col = f'team{team}_trump_risk'
        win_col = f'team{team}_win_rate'
        ax1.scatter(risk_analysis_data[risk_col], risk_analysis_data[win_col], 
                   alpha=0.5, label=f'Team {team}', s=50)
    ax1.set_xlabel('Trump Risk Factor', fontsize=12)
    ax1.set_ylabel('Win Rate', fontsize=12)
    ax1.set_title('Win Rate vs Trump Risk', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Win rate vs Gameplay Risk
    ax2 = axes[0, 1]
    for team in [0, 1]:
        risk_col = f'team{team}_gameplay_risk'
        win_col = f'team{team}_win_rate'
        ax2.scatter(risk_analysis_data[risk_col], risk_analysis_data[win_col], 
                   alpha=0.5, label=f'Team {team}', s=50)
    ax2.set_xlabel('Gameplay Risk Factor', fontsize=12)
    ax2.set_ylabel('Win Rate', fontsize=12)
    ax2.set_title('Win Rate vs Gameplay Risk', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Risk Difference vs Win Rate Difference
    ax3 = axes[1, 0]
    risk_analysis_data['win_rate_diff'] = abs(risk_analysis_data['team0_win_rate'] - risk_analysis_data['team1_win_rate'])
    ax3.scatter(risk_analysis_data['risk_difference'], risk_analysis_data['win_rate_diff'], 
               alpha=0.5, s=50)
    ax3.set_xlabel('Risk Factor Difference', fontsize=12)
    ax3.set_ylabel('Win Rate Difference', fontsize=12)
    ax3.set_title('Risk Difference vs Win Rate Difference', fontsize=14, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    
    # 4. Heatmap of average risk vs win rate
    ax4 = axes[1, 1]
    pivot_data = risk_analysis_data.groupby(['team0_avg_risk', 'team1_avg_risk'])['team0_win_rate'].mean().reset_index()
    if len(pivot_data) > 0:
        pivot_table = pivot_data.pivot(index='team0_avg_risk', columns='team1_avg_risk', values='team0_win_rate')
        sns.heatmap(pivot_table, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax4, cbar_kws={'label': 'Team 0 Win Rate'})
        ax4.set_xlabel('Team 1 Average Risk', fontsize=12)
        ax4.set_ylabel('Team 0 Average Risk', fontsize=12)
        ax4.set_title('Win Rate Heatmap', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\nVisualizations complete!")
else:
    print("No data available for visualization")


No data available for visualization
